In [ ]:
# set environment variables
import os

%set_env LOCAL_PROJECT_DIR=/dli/task/tao_project
%set_env LOCAL_DATA_DIR=/dli/task/flood_data

os.environ["LOCAL_EXPERIMENT_DIR"]=os.path.join(os.getenv("LOCAL_PROJECT_DIR"), "unet")

In [ ]:
# create directory for model
!mkdir -p models/flood_segmentation_model/1

# copy sample_resnet18.engine to the model repository
!cp $LOCAL_EXPERIMENT_DIR/export/sample_resnet18.engine models/flood_segmentation_model/1/model.plan

In [ ]:
configuration = """
name: "flood_segmentation_model"
platform: "tensorrt_plan"
max_batch_size: 1
input: [
 {
    name: "input_1"
    data_type: TYPE_FP32
    format: FORMAT_NCHW
    dims: [ 3, 512, 512 ]
  }
]
output: {
    name: "argmax_1"
    data_type: TYPE_INT32
    dims: [ 512, 512, 1 ]
  }
"""

with open('models/flood_segmentation_model/config.pbtxt', 'w') as file:
    file.write(configuration)

In [ ]:
# show model repository folder structure
!tree -a models

In [ ]:
# add time to wait for the model to be loaded in the Triton Inference Server
!sleep 45

In [ ]:
# To confirm Triton Inference Server is up and running
!curl -v triton:8000/v2/health/ready

In [ ]:
!curl -v triton:8000/v2/models/flood_segmentation_model

In [ ]:
# Pre-Process Inputs

# import dependencies
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import time
import warnings
import random

warnings.filterwarnings("ignore")

In [ ]:


# define pre-process function
def preprocess_image(image): 
    """
    This function returns the pre-processed image as a NumPy array. 
    """
    # load image as a 32-bit floating point array 
    image_ary=np.asarray(image)
    image_ary=image_ary.astype(np.float32)
    
    # pre-processing
    # y = net_scale_factor * (x-mean)
    # where, net_scale_factor = 1/127.5 and mean = 127.5
    
    image_ary=(image_ary-127.5)*0.00784313725490196
    
    # unet segmentation model requires the data to be in BGR format
    BGR=np.empty_like(image_ary)
    BGR[:, :, 0]=image_ary[:, :, 2]
    BGR[:, :, 1]=image_ary[:, :, 1]
    BGR[:, :, 2]=image_ary[:, :, 0]
    image_ary=BGR
    
    # convert array from h, w, c to 1, c, h, w
    image_ary=np.transpose(image_ary, [2, 0, 1])
    image_ary=np.expand_dims(image_ary, axis=0)
    return image_ary

In [ ]:
# choose random image
random_image_file=random.sample(os.listdir(os.path.join(os.getenv('LOCAL_DATA_DIR'), 'images', 'all_images')), 1)[0]

# preprocess
image=Image.open(os.path.join(os.getenv('LOCAL_DATA_DIR'), 'images', 'all_images', random_image_file))
mask=Image.open(os.path.join(os.getenv('LOCAL_DATA_DIR'), 'masks', 'all_masks', random_image_file))
image_ary=preprocess_image(image)

print('The input array has a shape of {}.'.format(image_ary.shape))

In [ ]:
import tritonclient.http as tritonhttpclient
from pprint import pprint

# set parameters
VERBOSE=False
input_name='input_1:0'
input_shape=(1, 3, 512, 512)
input_dtype='FP32'
output_name='argmax_1'
model_name='flood_segmentation_model'
url='triton:8000'
model_version='1'

In [ ]:
# instantiate Triton Inference Server client
triton_client=tritonhttpclient.InferenceServerClient(url=url, verbose=VERBOSE)

# get model metadata
print('----------Metadata----------')
model_metadata=triton_client.get_model_metadata(model_name=model_name, model_version=model_version)
pprint(model_metadata)

# get model configuration
print('----------Configuration----------')
model_config=triton_client.get_model_config(model_name=model_name, model_version=model_version)
pprint(model_config)

In [ ]:
inference_input=tritonhttpclient.InferInput(input_name, input_shape, input_dtype)
output=tritonhttpclient.InferRequestedOutput(output_name)

inference_input.set_data_from_numpy(image_ary)

# time the process
start=time.time()
response=triton_client.infer(model_name, 
                             model_version=model_version, 
                             inputs=[inference_input], 
                             outputs=[output])
latency=time.time()-start
logits=response.as_numpy(output_name)

print('The output array has a shape of {}.'.format(logits.shape))
print('It took {} per inference.'.format(round(latency, 3)))

In [ ]:
# visualize the output
# visualize results
fig, ax_arr=plt.subplots(1, 3, figsize=[15, 5], sharex=True, sharey=True)
ax_arr[0].set_title('Input Data')
ax_arr[1].set_title('Inference')
ax_arr[2].set_title('Actual')
ax_arr[0].set_xticks([])
ax_arr[0].set_yticks([])

ax_arr[0].imshow(image)
ax_arr[1].imshow(logits[0], cmap='gray')
ax_arr[2].imshow(mask, cmap='gray')

fig.tight_layout()
plt.show()

In [ ]:
time_list=[]

for image_path in os.listdir(os.path.join(os.getenv('LOCAL_DATA_DIR'), 'images', 'all_images')): 
    image=Image.open(os.path.join(os.getenv('LOCAL_DATA_DIR'), 'images', 'all_images', image_path))
    image_ary=preprocess_image(image)
    inference_input.set_data_from_numpy(image_ary)
    
    # time the process
    start=time.time()
    response=triton_client.infer(model_name, 
                                 model_version=model_version, 
                                 inputs=[inference_input], 
                                 outputs=[output])
    time_list.append(time.time()-start)
    logits=response.as_numpy(output_name)
    
latency=sum(time_list)/len(time_list)
print('It took {} seconds to infer {} images.'.format(round(sum(time_list), 3), len(time_list)))
print('On average it took {} seconds per inference.'.format(round(latency, 3)))

In [ ]:
# batch inference
batch_size=8
# create directory for model
!mkdir -p models/flood_segmentation_model_batch/1

# copy sample_resnet18.engine to the model repository
!cp $LOCAL_EXPERIMENT_DIR/export/sample_resnet18.engine models/flood_segmentation_model_batch/1/model.plan

In [ ]:
configuration = """
name: "flood_segmentation_model_batch"
platform: "tensorrt_plan"
max_batch_size: {}
input: [
 {{
    name: "input_1:0"
    data_type: TYPE_FP32
    format: FORMAT_NCHW
    dims: [ 3, 512, 512 ]
 }}
]
output: {{
    name: "argmax_1"
    data_type: TYPE_INT32
    dims: [ 512, 512, 1]
 }}
""".format(batch_size)

with open('models/flood_segmentation_model_batch/config.pbtxt', 'w') as file:
    file.write(configuration)

In [ ]:
# show model repository folder structure
!tree -a models

In [ ]:
# add time to wait for the model to be loaded in the Triton Inference Server
!sleep 45

In [ ]:
!curl -v triton:8000/v2/models/flood_segmentation_model_batch

In [ ]:
# define new input shape
batch_input_shape=(batch_size, 3, 512, 512)

batch_inference_input=tritonhttpclient.InferInput(name='input_1:0', shape=batch_input_shape, datatype='FP32')
batch_output=tritonhttpclient.InferRequestedOutput('argmax_1')

# create empty array for the batch input
batch_ary=np.empty(batch_input_shape).astype(np.float32)

time_list=[]

# iterate through all images
for idx, image_path in enumerate(os.listdir(os.path.join(os.getenv('LOCAL_DATA_DIR'), 'images', 'all_images'))): 
    image=Image.open(os.path.join(os.getenv('LOCAL_DATA_DIR'), 'images', 'all_images', image_path))
    batch_ary[idx%batch_size]=preprocess_image(image)
    if idx%batch_size==(batch_size-1): 
        batch_inference_input.set_data_from_numpy(batch_ary)

        # time the process
        start=time.time()
        response=triton_client.infer(model_name='flood_segmentation_model_batch', 
                                     model_version='1', 
                                     inputs=[batch_inference_input], 
                                     outputs=[batch_output])
        time_list.append(time.time()-start)
        logits=response.as_numpy(output_name)

batch_latency=sum(time_list)/len(time_list)
print('It took {} seconds to infer {} images.'.format(round(sum(time_list), 3), len(time_list)*batch_size))
print('On average it took {} seconds per inference.'.format(round(batch_latency, 3)))

In [ ]:
# plot throughput vs. latency
plt.title('Inference Performance Comparison')
plt.plot([latency, batch_latency], [1/latency, batch_size/batch_latency], marker='o')
plt.text(latency, (1/latency)-2.5, 'Non-Batching')
plt.text(batch_latency, (batch_size/batch_latency)-2.5, 'Batching ({})'.format(batch_size))

plt.xlabel('Latency (Second)')
plt.ylabel('Throughput (Image/Second)')
plt.xlim(xmin=0, xmax=max(batch_latency, latency)*1.25)
plt.ylim(ymin=0, ymax=max(1/latency, batch_size/batch_latency)*1.25)
plt.show()